# Module 1 Final Project Demo

Run the course-style Personal Chef assistant and its Conference Rehearsal adaptation with Gemini, Tavily, and thread-scoped memory.

In [1]:
import html
import json
import re
import sys
from pathlib import Path
from pprint import pprint
from urllib.parse import urlsplit

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
import requests

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    project_root = project_root.parent

load_dotenv(project_root / '.env')
expected_python = (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
assert Path(sys.executable).resolve() == expected_python
assert sys.version_info[:3] == (3, 13, 1)
project_dir = project_root / 'my-work' / 'module1_final_project'
sys.path.insert(0, str(project_dir))
from chef_agent import create_chef_agent, thread_config as chef_thread_config
from conference_rehearsal_agent import create_conference_rehearsal_agent, thread_config as conference_thread_config

def show_messages(label, messages):
    print(f'\n===== {label} =====')
    for index, message in enumerate(messages, start=1):
        if isinstance(message, HumanMessage):
            role = 'HUMAN MESSAGE'
        elif isinstance(message, AIMessage) and message.tool_calls:
            role = 'AI TOOL CALL'
        elif isinstance(message, ToolMessage):
            role = 'TAVILY TOOL RESULT' if message.name == 'web_search' else f'TOOL RESULT: {message.name}'
        elif isinstance(message, AIMessage):
            role = 'FINAL ANSWER'
        else:
            role = type(message).__name__.upper()
        print(f'\n--- Message {index}: {role} ---')
        pprint(message)

def print_complete_state(agent, config, label):
    snapshot = agent.get_state(config)
    print(f'\n===== COMPLETE STORED STATE: {label} =====')
    pprint(snapshot.values)
    print('Stored message count:', len(snapshot.values.get('messages', [])))
    return snapshot.values

print('Using project .venv, Python 3.13.1, Gemini Flash-Lite, and the existing Tavily setup.')

Using project .venv, Python 3.13.1, Gemini Flash-Lite, and the existing Tavily setup.


## Personal Chef assistant

In [2]:
chef_checkpointer = InMemorySaver()
chef_agent = create_chef_agent(checkpointer=chef_checkpointer)
chef_config = chef_thread_config('chef-demo-abdelmalek-1')
chef_first_message = 'I have leftover chicken, cooked rice and carrots. I have 25 minutes and no oven. Find something practical I can prepare.'
chef_first_result = chef_agent.invoke(
    {'messages': [HumanMessage(content=chef_first_message)]},
    config=chef_config,
)
show_messages('Chef: first request', chef_first_result['messages'])


===== Chef: first request =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='I have leftover chicken, cooked rice and carrots. I have 25 minutes and no oven. Find something practical I can prepare.', additional_kwargs={}, response_metadata={}, id='d502427e-5740-49f4-8e82-80ef2f1b5aa9')

--- Message 2: AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "quick recipes with leftover chicken cooked rice and carrots no oven"}'}, '__gemini_function_call_thought_signatures__': {'call_390412': 'EnMKcQFpFH0TGg1abJk4IQUowANLUZ60pZw0v2HCG0a0BRZYASOuyfR1+N3UCugX1LoUMC7Lbwma+Q6Bl+nuAfQ+dGvs4tjpNzsxlnrjNLUdwsKqs/VeySllY8H23t5bn2jflonoA5Guk9bePOzdr5+IssIA'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fe0d-f970-7ac3-ab1e-28118acb7c0c-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'quick recipes wit

In [3]:
chef_follow_up = 'Make it dairy-free and remember that I prefer spicy food.'
chef_second_result = chef_agent.invoke(
    {'messages': [HumanMessage(content=chef_follow_up)]},
    config=chef_config,
)
show_messages('Chef: dairy-free and spicy follow-up', chef_second_result['messages'])
chef_state = print_complete_state(chef_agent, chef_config, 'Chef thread')
chef_human_text = ' '.join(message.content for message in chef_state['messages'] if isinstance(message, HumanMessage))
print('Same-thread ingredient and restriction history present:', all(term in chef_human_text.lower() for term in ('chicken', 'rice', 'carrots', 'dairy-free', 'spicy')))


===== Chef: dairy-free and spicy follow-up =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='I have leftover chicken, cooked rice and carrots. I have 25 minutes and no oven. Find something practical I can prepare.', additional_kwargs={}, response_metadata={}, id='d502427e-5740-49f4-8e82-80ef2f1b5aa9')

--- Message 2: AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "quick recipes with leftover chicken cooked rice and carrots no oven"}'}, '__gemini_function_call_thought_signatures__': {'call_390412': 'EnMKcQFpFH0TGg1abJk4IQUowANLUZ60pZw0v2HCG0a0BRZYASOuyfR1+N3UCugX1LoUMC7Lbwma+Q6Bl+nuAfQ+dGvs4tjpNzsxlnrjNLUdwsKqs/VeySllY8H23t5bn2jflonoA5Guk9bePOzdr5+IssIA'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fe0d-f970-7ac3-ab1e-28118acb7c0c-0', tool_calls=[{'name': 'web_search', 'args': {'query': '

## Conference Rehearsal assistant

In [4]:
conference_checkpointer = InMemorySaver()
conference_agent = create_conference_rehearsal_agent(checkpointer=conference_checkpointer)
conference_config = conference_thread_config('conference-demo-abdelmalek-1')
conference_first_message = 'My name is Abdelmalek. I am preparing an eight-minute conference presentation about vehicle tyre-pressure monitoring using machine learning. The audience includes both technical and non-technical attendees.'
conference_first_result = conference_agent.invoke(
    {'messages': [HumanMessage(content=conference_first_message)]},
    config=conference_config,
)
show_messages('Conference: presentation context', conference_first_result['messages'])


===== Conference: presentation context =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='My name is Abdelmalek. I am preparing an eight-minute conference presentation about vehicle tyre-pressure monitoring using machine learning. The audience includes both technical and non-technical attendees.', additional_kwargs={}, response_metadata={}, id='7c56809d-0834-450f-8b11-8a652e5945af')

--- Message 2: FINAL ANSWER ---
AIMessage(content=[{'type': 'text', 'text': 'Hello, Abdelmalek. It’s a pleasure to assist you. An eight-minute slot is very tight—it usually allows for only **5 to 6 core slides** and requires a very disciplined structure to avoid running over time.\n\nTo help you get started, I have outlined a strategy tailored for a mixed-expertise audience, balancing your technical machine learning (ML) methodology with the practical, high-level impact of your research.\n\n### 1. The Recommended Structure (8-Minute Timeline)\n\n| Section | Time | Goal |\n| :--- | :--- | :--- |

In [5]:
conference_source_request = 'Find two current credible sources explaining why external validation matters in machine-learning systems, then suggest where this idea could fit in my presentation.'
conference_source_result = conference_agent.invoke(
    {'messages': [HumanMessage(content=conference_source_request)]},
    config=conference_config,
)
show_messages('Conference: current external sources requested', conference_source_result['messages'])
conference_state_after_sources = print_complete_state(conference_agent, conference_config, 'Conference thread after source request')


===== Conference: current external sources requested =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='My name is Abdelmalek. I am preparing an eight-minute conference presentation about vehicle tyre-pressure monitoring using machine learning. The audience includes both technical and non-technical attendees.', additional_kwargs={}, response_metadata={}, id='7c56809d-0834-450f-8b11-8a652e5945af')

--- Message 2: FINAL ANSWER ---
AIMessage(content=[{'type': 'text', 'text': 'Hello, Abdelmalek. It’s a pleasure to assist you. An eight-minute slot is very tight—it usually allows for only **5 to 6 core slides** and requires a very disciplined structure to avoid running over time.\n\nTo help you get started, I have outlined a strategy tailored for a mixed-expertise audience, balancing your technical machine learning (ML) methodology with the practical, high-level impact of your research.\n\n### 1. The Recommended Structure (8-Minute Timeline)\n\n| Section | Time | Goal |\n| :--- |

In [6]:
def message_text(message):
    content = message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return ' '.join(block.get('text', '') for block in content if isinstance(block, dict))
    return str(content)

source_records = []
for message in conference_source_result['messages']:
    if isinstance(message, ToolMessage) and message.name == 'web_search':
        payload = json.loads(message.content) if isinstance(message.content, str) else message.content
        source_records.extend(payload.get('results', []))
for message in conference_source_result['messages']:
    if isinstance(message, AIMessage):
        for url in re.findall(r'https?://[^\s)\]}>\"\']+', message_text(message)):
            source_records.append({'url': url.rstrip('.,;'), 'title': 'URL cited by assistant', 'content': 'citation'})

unique_sources = {}
for source in source_records:
    url = source.get('url')
    if url and urlsplit(url).scheme in {'http', 'https'}:
        unique_sources.setdefault(url, source)

source_checks = []
for url, source in unique_sources.items():
    record = {'url': url, 'search_title': source.get('title'), 'search_snippet': source.get('content', '')[:400]}
    try:
        page = requests.get(url, timeout=12, headers={'User-Agent': 'Mozilla/5.0'})
        record['http_status'] = page.status_code
        record['final_url'] = page.url
        record['reachable'] = page.ok
        page_text = html.unescape(re.sub(r'<[^>]+>', ' ', page.text)).lower()
        record['page_mentions_machine_learning'] = 'machine learning' in page_text or 'machine-learning' in page_text
        record['page_mentions_validation'] = 'validation' in page_text or 'validating' in page_text
        record['topic_terms_found'] = record['page_mentions_machine_learning'] and record['page_mentions_validation']
        record['credible_source_verified'] = False
    except requests.RequestException as error:
        record['reachable'] = False
        record['verification_error'] = f'{type(error).__name__}: {error}'
        record['credible_source_verified'] = False
    source_checks.append(record)

print('URL checks for every Tavily result and assistant citation:')
pprint(source_checks)
print('A reachable URL and matching topic terms are only preliminary checks, not a credibility endorsement.')

URL checks for every Tavily result and assistant citation:
[{'credible_source_verified': False,
  'final_url': 'https://www.sciencedirect.com/science/article/abs/pii/S016926072100362X',
  'http_status': 403,
  'page_mentions_machine_learning': False,
  'page_mentions_validation': False,
  'reachable': False,
  'search_snippet': 'The external validation of ML models is increasingly '
                    'being proposed as the main (and only) means to certify '
                    'the supposed validity of the model on (virtually any) '
                    'unseen data, . However, as also the findings shown above '
                    'illustrate, the result from an external validation cannot '
                    'guarantee reliability _per se_: if the external '
                    'validation had been performed only on the Italy-2 '
                    'dataset, where the diagno',
  'search_title': 'The importance of being external. methodological insights '
                  'for the

In [7]:
conference_follow_up = 'Now reduce your presentation suggestion to three concise bullet points and remember my eight-minute limit.'
conference_final_result = conference_agent.invoke(
    {'messages': [HumanMessage(content=conference_follow_up)]},
    config=conference_config,
)
show_messages('Conference: concise rehearsal follow-up', conference_final_result['messages'])
conference_state = print_complete_state(conference_agent, conference_config, 'Conference thread final state')
conference_human_text = ' '.join(message.content for message in conference_state['messages'] if isinstance(message, HumanMessage))
print('Same-thread topic, time limit, and audience retained:', all(term in conference_human_text.lower() for term in ('tyre-pressure', 'eight-minute', 'non-technical')))


===== Conference: concise rehearsal follow-up =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='My name is Abdelmalek. I am preparing an eight-minute conference presentation about vehicle tyre-pressure monitoring using machine learning. The audience includes both technical and non-technical attendees.', additional_kwargs={}, response_metadata={}, id='7c56809d-0834-450f-8b11-8a652e5945af')

--- Message 2: FINAL ANSWER ---
AIMessage(content=[{'type': 'text', 'text': 'Hello, Abdelmalek. It’s a pleasure to assist you. An eight-minute slot is very tight—it usually allows for only **5 to 6 core slides** and requires a very disciplined structure to avoid running over time.\n\nTo help you get started, I have outlined a strategy tailored for a mixed-expertise audience, balancing your technical machine learning (ML) methodology with the practical, high-level impact of your research.\n\n### 1. The Recommended Structure (8-Minute Timeline)\n\n| Section | Time | Goal |\n| :--- | :--- |

## Observations

- What the Chef assistant does: Searches for practical recipes using the available ingredients and constraints. Its first Tavily result supported a 15-minute stovetop leftover-chicken fried rice recipe. The agent cited a URL variant that did not verify; the exact Tavily URL did.

- What the Conference Rehearsal assistant does: Remembers the topic, eight-minute limit, and mixed audience; it produced an outline, searched when asked for current evidence, and condensed the suggestion to three bullets.

- Evidence that memory worked: The Chef thread stored six messages, including both user turns and the intervening AI/tool messages. The Conference thread stored eight messages and the final answer retained the eight-minute limit and audience context.

- Evidence that web search was used: Both agents returned an AI `web_search` tool call and a Tavily `ToolMessage`. The URL-check cell examined every returned result URL and every URL cited by the Conference answer.

- What each agent got wrong or handled poorly: The Chef asserted a dairy-free adaptation without checking every recipe ingredient and altered the source URL in its citation. The Conference assistant called its citations credible although the ScienceDirect page redirected to sign-in, the cited Domino URL returned a not-found page on fetch, and the accessible search snippets were insufficient to verify both claims. It also assumed the user had impressive/high internal accuracy results and called external validation an industry gold standard without evidence from the user or verified sources. The 2025 PMC result was not selected by the assistant and its page presented an anti-bot challenge, so it was not independently verified here.

- One improvement I would make next: Require the Chef to compare an actual recipe ingredient list against dietary restrictions and preserve the exact returned URL. For Conference, prefer accessible peer-reviewed or official sources, verify the article text before citing it, and forbid any reference to unprovided experimental results.